# 课程 3 · 帮 IT 同事整理一句报修消息

假设你在一家小公司帮忙。员工电脑出了问题，会给负责电脑和网络的 IT 同事发消息。IT 同事每次都要读一遍、记下问题，缺信息时再追问。消息多了，这件事就很重复。

今天我们试着替他做其中一小步：**把员工的一句报修话，整理成一份待确认的登记记录。** 平时大家把这种问题记录叫“工单”，本课先叫它“登记记录”。

## 我们手里有什么

| 手里的东西 | 从哪里来、能做什么 |
|---|---|
| 能运行 Python 的电脑 | 前两课已经准备好，可以存文字、做判断、调用函数 |
| 一个训练过的小模型 | 上一课用英文名字训练的 bigram，只会根据前一个字符猜下一个字符 |
| 一条员工消息 | 本课编的一条练习消息：下面会亲手把它存进变量 |
| 一份登记要求 | 我们为这次练习自己定，下面会逐项说明 |

公司工单系统、内部资料库还没有接上。今天的结果会显示在 notebook 里，供人检查；后面的课再学习怎样保存到业务系统。

## 今天希望得到什么

员工说：**“我的 VPN 连不上，请帮我创建一张 IT 工单。”** VPN 在这里是连接公司内网的工具。

如果由人来整理，可以写成这样：

| 登记项目 | 人工填写的示例 | 为什么要记 |
|---|---|---|
| 问题类别 | 网络连接 | 方便交给负责这类问题的人 |
| 问题简介 | VPN 无法连接 | 让接手的人迅速知道发生了什么 |
| 还需要问什么 | 操作系统、错误提示 | 原话没说清这些，后续排查需要补充 |

**这张表是我们人工填写的目标示例，还没调用模型。** 为了让练习有明确标准，我们约定：连接网络的问题先记为网络类；明确的账号、密码问题记为账号类；判断不了记为其他。遇到 VPN 连接故障，先收集操作系统和错误提示。这是在规定怎样登记，不是在诊断真实故障原因。

这一课从这句原话出发，最后让 Python 显示同样三项信息，并告诉我们还需要追问什么。

## 开始前 · 用哪个 Python 运行

可以选择你界面上的 **`llm-from-zero` 内核**。内核的名字由人命名，关键是它使用哪个 Python。运行下面两行查看；这个项目的环境路径应以 `llm-from-zero/.venv/bin/python` 结尾。

`.venv` 是环境所在的文件夹，不要求内核在界面上也叫这个名字。

In [1]:
import sys
print(sys.executable)

/Users/yorkchen/文稿 - 陈尧的MacBook Pro/VibeCoding/llm-from-zero/.venv/bin/python


## 第 1 步 · 先把员工原话交给程序

**现在的问题**：我们能读懂这句话，Python 还没有拿到它。

**这一步做什么**：把它存成一个字符串。这和前面存英文名字一样，文字放在引号里，变量 `user_text` 指向这段文字。直接运行。

In [2]:
user_text = "我的 VPN 连不上，请帮我创建一张 IT 工单。"
print(user_text)

我的 VPN 连不上，请帮我创建一张 IT 工单。


现在程序能保存并打印原话了。但打印原话不等于整理出“网络连接、VPN 无法连接、还缺什么”。

**接下来要解决**：怎样让程序判断这句话属于哪类问题？先试一个我们能写出来的简单办法。

## 第 2 步 · 看到 VPN，就当作网络问题，行不行

**现在的问题**：程序只有文字，没有分类办法。

**这一步做什么**：先写一条规则——出现 `VPN` 就返回 `network`，否则返回 `other`。这两个英文词分别代表“网络类”“其他”。

**本步的 Python 写法**：`"VPN" in text` 检查字符串里是否包含这一小段文字，结果是 `True` 或 `False`。先直接运行，看它能不能解决我们的问题。

In [3]:
def guess_category(text):
    if "VPN" in text:
        return "network"
    return "other"

print(guess_category(user_text))

network


第一句返回 `network`，按本课约定分对了。再换两种说法，代码不用改。

In [4]:
print(guess_category("公司内网连不上了。"))
print(guess_category("VPN 已经能用了，现在是邮箱密码忘了。"))

other
network


会看到：

- 第一条返回 `other`：没写 VPN，就漏掉了网络问题。
- 第二条返回 `network`：明明当前要处理的是邮箱密码，它还是盯住了 VPN。

我们当然可以继续加规则；表达少、规则稳定时，这就可能够用。这里要练的是员工用不同说法描述问题，程序还要整理简介、找出没交代的信息，所以接下来尝试让一个现成模型帮忙。

**上一课的小模型为什么不能直接用？** 它的词表只有英文字母和结束符，训练数据是名字，而且只看前一个字符。这一课需要处理整句中文报修，超出了它学过的内容和结构。

**接下来要解决**：我们会借用已经训练好的大模型。不过，先要说明让它替我们填什么，回答怎样写才方便 Python 读取。

## 第 3 步 · 为什么要约定回答格式

**现在的问题**：就算模型能理解句子，它也可能只回一段话。

例如下面这句是我手写的回答示例：

> 看起来是网络连接问题。可以先问一下操作系统和错误提示。

人读起来没问题，但 Python 接下来想分别拿到“类别”“简介”“还缺什么”。每次回答的句式都变化，程序还得再想办法从句子里找这三项。

**这一步做什么**：把开头那张三栏表，写成双方约定的固定格式。

| 中文项目 | 代码里约定的名字 | 本课允许怎样填 |
|---|---|---|
| 问题类别 | `category` | `network` 网络类、`account` 账号类、`other` 其他 |
| 问题简介 | `summary` | 一段不超过 80 字符的简短文字 |
| 还需要问什么 | `missing_fields` | 一个列表：每项是需要追问的信息；齐全时用空列表 `[]` |

这些名字是我们选的，模型不会自动知道，下一步要在任务说明里告诉它。

把这三项写出来就是：

```json
{
  "category": "network",
  "summary": "VPN无法连接",
  "missing_fields": ["操作系统", "错误提示"]
}
```

这种用固定规则写数据的文本格式叫 **JSON**。你可以先把它看成“给每项信息起了名字的登记表”。稍后 Python 能按名字取值。

**接下来要解决**：把“要做什么、按什么格式回答”和员工原话一起交给模型。

## 第 4 步 · 把任务说明和员工原话放在一起（TODO 1）

**现在的问题**：登记要求在我们脑子里，模型还没收到。

**这一步做什么**：准备两条消息，一条说明要求，一条放员工原话。

一条消息用 Python 字典写：

```python
{"role": "user", "content": "你好"}
```

- `role` 表示这条消息的身份：`system` 放我们给助手的要求，`user` 放这次员工原话。
- `content` 放文字内容。
- 两条消息放进一个列表，叫 `messages`，意思就是“这些消息”。

下方 `rules` 是任务说明，已经按第 3 步的表写好了。阅读它，确认每个要求都知道来历。你只需把 `None` 换成第 1 步存好的 `user_text`。

注意：`user_text` 是变量，`"user_text"` 只是几个英文字母组成的文字。

<details><summary>提示 1</summary>

第二条消息应该放员工说的那句话。

</details>

<details><summary>提示 2</summary>

那句话已经存进 user_text，直接把变量放进去。

</details>

<details><summary>答案</summary>

```python
{"role": "user", "content": user_text}
```

</details>

In [6]:
rules = """请帮 IT 同事整理员工的报修消息，生成一份待确认的登记记录，只输出 JSON。
只填三项：category、summary、missing_fields。
category：连接网络的问题填 network；明确的账号密码问题填 account；判断不了填 other。
summary：用 1–80 字符概括当前要处理的问题。
missing_fields：需要继续追问的信息名称列表，已齐全时填 []。
本课规定：VPN 连接故障要收集操作系统和错误提示，原话没给就列入 missing_fields，不要编造。
格式示例：{"category":"other","summary":"用户描述的问题","missing_fields":[]}
你负责整理信息，不要声称已经创建工单或修好问题。"""
messages = [
    {"role": "system", "content": rules},
    {"role": "user", "content": user_text},  # TODO 1：放入员工原话
]
assert messages[1]["content"] == user_text, "先完成 TODO 1"
print(messages[1])

{'role': 'user', 'content': '我的 VPN 连不上，请帮我创建一张 IT 工单。'}


现在 `messages` 是准备发出的两条消息。运行这一格只在本机存数据，模型还没收到。

**接下来要解决**：模型在哪里，Python 怎样把这些消息送过去？

## 第 5 步 · 把消息送给模型

**现在的问题**：我们的电脑上没有运行能处理这条中文消息的大模型。

**这一步做什么**：使用服务商已经运行起来的模型。我们的程序通过网络发送消息，服务商的程序运行模型，再把生成的回答返回来。这个给程序使用的入口叫 **API**。

```text
这台电脑上的 Python                     服务商的电脑
       messages ─────── 通过 API ───────→ 模型处理消息
       response ←────── 返回结果 ──────── 生成回答
```

API key 是调用账号的凭证，模型名说明要用哪个模型。具体配置放在 [README](README.md)，需要真实调用时再照着做。

为了把今天的注意力放在“发什么、收到后怎么用”，负责发送消息的代码已经写在 `llm_client.py`，你用一个函数就能调用它：

| 函数 | 它具体做什么 |
|---|---|
| `call_model(messages)` | 通过网络发送这两条消息，等待并返回模型服务的结果 |
| `load_example_response()` | 读取本地的一份人工样例，供我们练习拿出回答、读出各项信息 |

**两者的结果来历不同。** 人工样例是为这条 VPN 消息写好的，不会理解新问题，也不会随提示词改变。它能帮你检查后半段 Python 代码是否能工作；模型能否判断正确，需要真实调用后观察。

下面这格让 notebook 找到这些函数，也导入稍后读取 JSON 要用的工具。可以直接运行；它只加载函数，不发送消息。

In [7]:
from pathlib import Path
import sys
import json

root = Path.cwd()
if not (root / "pyproject.toml").exists():
    root = root.parent
lesson_dir = root / "03-llm-api"
if str(lesson_dir) not in sys.path:
    sys.path.insert(0, str(lesson_dir))
from llm_client import load_example_response, call_model, validate_ticket
print("准备完成。默认使用手写教学样例，无模型调用费用。")

准备完成。默认使用手写教学样例，无模型调用费用。


现在选择结果从哪里来：

- 保持 `USE_LIVE_API = False`：读人工样例，无须账号、不会产生调用费。
- 配置好账号后改成 `True`：将 `messages` 发给模型，得到本次真实回答；每运行本格一次就会请求一次。

这次先把完整过程学明白，选择人工样例也能继续。真实调用遇到账号或网络问题时，会直接显示问题，不会暗中换成人工答案。

In [8]:
USE_LIVE_API = False
if USE_LIVE_API:
    response = call_model(messages)
    print("当前来源：真实 API 响应（本格发生一次请求）")
else:
    response = load_example_response()
    print("当前来源：手写教学样例（无模型调用）")

当前来源：手写教学样例（无模型调用）


变量 `response` 现在装着返回的内容。先直接打印，观察有哪些名字，不要求一次记住全部。

In [9]:
print(response)

{'_lesson_note': '手写教学样例，不是实测模型响应；usage 数字也是假设值。', 'model': 'handwritten-example-not-a-model', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '{"category":"network","summary":"VPN无法连接","missing_fields":["操作系统","错误提示"]}'}, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 180, 'completion_tokens': 45, 'total_tokens': 225}}


返回结果除了回答文字，还带着生成结束原因、输入输出用了多少等信息。我们先找回答正文。

**接下来要解决**：怎样从这份返回结果里拿出模型写的那段文字？

## 第 6 步 · 拿出回答正文（TODO 2）

**现在的问题**：`response` 里面既有回答，也有其他信息。

**这一步做什么**：按服务约定的结构，一层层找到正文。

```text
response
  └─ ["choices"]        回答列表
       └─ [0]            第一份回答，本课只取这一份
            └─ ["message"]
                 └─ ["content"]   回答正文
```

`response["choices"]` 按字典的名字取值，`[0]` 按列表位置取第一个元素。这里不是上一课 Tensor 的配对索引，只是接连取了几次值。

先把第一份回答取出来，存成 `choice`。你接着从 `choice` 取出 `message` 里的 `content`。

另一个名字 `finish_reason` 说明回答为什么结束。`stop` 表示正常结束；本课先检查这个，避免把写到一半的回答当成完整记录。

<details><summary>提示 1</summary>

choice 已经是第一份回答，不需要再找 choices。

</details>

<details><summary>提示 2</summary>

还剩两层：message，然后 content。

</details>

<details><summary>答案</summary>

```python
content = choice["message"]["content"]
```

</details>

In [11]:
choice = response["choices"][0]
print("结束原因：", choice["finish_reason"])
assert choice["finish_reason"] == "stop", "本次未正常完成，请检查输出上限与返回内容"
content = choice["message"]["content"]  # TODO 2：从 choice 取 message 里的 content
assert content is not None, "先完成 TODO 2"
assert isinstance(content, str) and content.strip(), "正文为空；检查模型返回和提示词"
print(content)
print("正文类型：", type(content))

结束原因： stop
{"category":"network","summary":"VPN无法连接","missing_fields":["操作系统","错误提示"]}
正文类型： <class 'str'>


现在可以直接看到登记内容了。但打印出的类型是 `str`：它仍然是一段文字。

**接下来要解决**：怎样让 Python 直接取出 `category`，而不是继续面对整段文字？

## 第 7 步 · 把这段文字读成 Python 数据（TODO 3）

**现在的问题**：回答长得像字典，实际仍是字符串。

**这一步做什么**：用 `json.loads()` 按 JSON 规则读它，得到 Python 数据。

先看一个更小的例子：

```python
person = json.loads('{"name": "小陈"}')
print(person["name"])  # 得到 小陈
```

外面的单引号说明输入是一段文字。`json.loads()` 读出其中的对象，变成字典，之后就能按 `"name"` 取值。同理，它会把 JSON 数组读成 Python 列表。

下面将 `content` 交给同一个函数，把结果存入 `ticket`。`ticket` 就是这份登记记录的变量名。

<details><summary>提示 1</summary>

这里需要把文字读成数据，用 loads。

</details>

<details><summary>提示 2</summary>

json.loads 的输入是上一格取出的 content。

</details>

<details><summary>答案</summary>

```python
ticket = json.loads(content)
```

</details>

In [12]:
ticket = json.loads(content)  # TODO 3：解析 content
assert ticket is not None, "先完成 TODO 3"
print("解析后类型：", type(ticket))
print(ticket)

解析后类型： <class 'dict'>
{'category': 'network', 'summary': 'VPN无法连接', 'missing_fields': ['操作系统', '错误提示']}


现在类型应是 `dict`。对照一下我们走过的路径：

| 变量 | 里面是什么 | 从哪里来 |
|---|---|---|
| `user_text` | 员工原话 | 第 1 步写入 |
| `messages` | 任务要求和原话 | 第 4 步组装 |
| `response` | 完整返回结果 | 第 5 步真实请求，或读取人工样例 |
| `content` | 回答正文，仍是文字 | 第 6 步从返回结果取出 |
| `ticket` | Python 可以按名称取值的登记记录 | 这一格用 `json.loads()` 读出来 |

如果 JSON 文字写坏了，`json.loads()` 会报错；此时还没有得到可用记录。实际模型的回答也需要经过这一步。

**接下来要解决**：读成字典后，有没有漏项、填错名字或取值？

## 第 8 步 · 检查填得怎样，再决定问什么

**现在的问题**：Python 能读这份记录，不代表模型已经按我们的要求填好。

先看一个具体麻烦：我们约定网络类填 `network`，它却写成了中文 `网络`。人看得懂，后面按 `network` 分配的程序就对不上了。还有可能漏掉简介，或者把“要追问的信息列表”写成一整句话。

**这一步做什么**：把第 3 步的三栏要求逐项检查。负责这件事的 `validate_ticket(ticket)` 已经写好：有问题就返回问题列表，没有发现这些问题就返回 `[]`。

直接运行。这里的检查只针对三项信息是否齐全、类型和取值是否符合约定。

In [13]:
issues = validate_ticket(ticket)
print("字段检查问题：", issues)
if len(issues) == 0:
    print("分类：", ticket["category"])
    print("摘要：", ticket["summary"])
    print("还缺：", ticket["missing_fields"])

字段检查问题： []
分类： network
摘要： VPN无法连接
还缺： ['操作系统', '错误提示']


再看两份我们故意填得不对的记录。第一份把分类写成中文；第二份虽然用的是允许的英文词，却把本课这条 VPN 消息记为账号类，还漏掉了应追问的信息。

In [14]:
bad_schema = {"category": "网络", "summary": "VPN无法连接", "missing_fields": []}
print("JSON 合法，但分类不在约定内：", validate_ticket(bad_schema))

wrong_meaning = {"category": "account", "summary": "VPN无法连接", "missing_fields": []}
print("字段检查可能通过，事实仍可能错误：", validate_ticket(wrong_meaning))

JSON 合法，但分类不在约定内： ['category 必须是 network / account / other 之一']
字段检查可能通过，事实仍可能错误： []


第二份的检查结果可能是 `[]`。因为“是不是允许的分类词”可以按规则检查，“是否准确表达了员工的意思、该问的是否都问了”还需要对照原话和登记要求。

因此我们先看两件事：**程序能否读取、各项是否符合约定；内容是否确实正确。** 后一件事，下节课会拿更多报修说法来观察。

现在回到我们收到的 `ticket`：

- 有填写问题，先处理这些问题。
- 还有信息没收集到，下一步向员工追问。
- 信息齐全，再核对内容，交给之后的处理步骤。

下面先把这个决定打印出来。

In [15]:
issues = validate_ticket(ticket)
if len(issues) > 0:
    print("下一步：先处理返回字段问题", issues)
elif len(ticket["missing_fields"]) > 0:
    print("下一步：向用户追问", ticket["missing_fields"])
else:
    print("下一步：核对内容，再交给后续业务处理环节")

下一步：向用户追问 ['操作系统', '错误提示']


人工样例应打印出需要追问“操作系统、错误提示”。现在能从回答里单独拿出分类、简介和追问项。若使用人工样例，今天跑通的是收到回答后的这部分程序；真实模型能否把原话整理好，仍需实际调用后观察。

**还没有完成的事也很具体**：程序没有向员工真的发出追问，没有把记录写进公司系统，也没有修好 VPN。后面的课会学习怎样发出追问、查询和保存登记记录；实际排障交给负责的同事处理。

这一课也给我们留下了一个问题：真模型遇到换一种说法、信息不全或多个问题时，能不能填好？下一课会准备几条不同的消息，逐条记录它分对了什么、错在哪里。

## 用自己的话说一遍

先不看代码，试着回答：

1. 我们最开始有什么？希望帮 IT 同事省下哪一步？
2. 为什么“看到 VPN 就分类”的办法会出错？这是否意味着所有任务都必须用大模型？
3. 为什么要约定 JSON？为什么读成字典后还要检查？
4. 你这次得到的回答来自真实模型，还是人工样例？这分别能说明什么？

**我的回答：**

（在这里写）

到这里可以结束本次学习。下面算费用是加做内容，不影响你理解刚才的过程。

## 加做 · 如果每天处理很多条，一次调用花多少钱（TODO 4）

**新问题**：如果以后每条消息都真正调用模型，每天有很多条，会花多少钱？

**这一步做什么**：回到 `response`，看看这次输入和输出用了多少，再乘单价。

模型处理文字的单位叫 **token**，不固定等于一个汉字或一个英文单词。今天先读取服务返回的数量，之后再专门观察它怎样切分文字。

这类 API 把用量放在 `usage` 中：`prompt_tokens` 是输入量，`completion_tokens` 是输出量。人工样例里的数字也是人为设定的，不是实测。

假设普通输入 **2 元 / 百万 token**，输出 **8 元 / 百万 token**，没有缓存和其他计费项。这是练习价格。

```text
费用 = 输入数量 / 1,000,000 × 输入单价
     + 输出数量 / 1,000,000 × 输出单价
```

你只需用乘除和加法填 `cost`。

<details><summary>提示 1</summary>

输入和输出单价不同，要分开算。

</details>

<details><summary>提示 2</summary>

两个数量都先除以一百万，再分别乘 2 和 8。

</details>

<details><summary>答案</summary>

```python
cost = input_tokens / 1_000_000 * 2 + output_tokens / 1_000_000 * 8
```

</details>

In [16]:
usage = response["usage"]
input_tokens = usage["prompt_tokens"]
output_tokens = usage["completion_tokens"]
print("输入 / 输出 token：", input_tokens, output_tokens)
cost = input_tokens / 1_000_000 * 2 + output_tokens / 1_000_000 * 8
assert cost is not None, "先完成 TODO 4"
print(f"假设价格下的调用费：{cost:.6f} 元")
print("数据来源：", "真实 API 用量" if USE_LIVE_API else "手写假设用量，不是实测")

输入 / 输出 token： 180 45
假设价格下的调用费：0.000720 元
数据来源： 手写假设用量，不是实测


真实调用代码中把一次最多生成的数量设为 512。这个上限限制最多生成多少，不表示实际一定用了 512；本例按返回的用量估算。

一个任务如果需要两次模型调用，就分别算费用，再相加。第二次输入如果带上了历史，历史也属于那次输入的一部分。真实账单还应按所用服务的计费规则对照；这里只练普通输入、输出两项的计算。

## 查阅 · 让打印结果更容易读

如果 `print(response)` 挤成一行不好看，可以运行：

```python
print(json.dumps(response, ensure_ascii=False, indent=2))
```

- `json.dumps()`：把 Python 数据写成 JSON 文字，方便查看。
- `ensure_ascii=False`：中文直接显示，不写成 `\u...` 的转义形式。
- `indent=2`：换行，每深入一层缩进两个空格。

它们只改变怎样打印，不会让模型的回答发生变化。注意 `dumps` 是数据变文字，刚才的 `loads` 是文字变数据。

课后作业见 [homework.ipynb](homework.ipynb)，完成后说“批改”。其中费用题对应本课加做部分。

查文档时再用这些链接：[首次调用](https://api-docs.deepseek.com/quick_start) · [请求与返回](https://api-docs.deepseek.com/api/create-chat-completion/) · [JSON 输出](https://api-docs.deepseek.com/guides/json_mode/)（本项目 2026-10-30 核实）。